In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from relbench.datasets import get_dataset
from relbench.tasks import get_task

import utils

%load_ext autoreload
%autoreload 2

# graphs too large for interactivity
pio.renderers.default = 'svg'

In [ ]:
dataset = get_dataset(name='rel-stack', download=True)

In [ ]:
dataset.get_db().table_dict.keys()

#### Schema:
[Full Schema](https://meta.stackexchange.com/questions/2677/database-schema-documentation-for-the-public-data-dump-and-sede/2678#2678)

![|500](https://relbench.stanford.edu/img/stack-exchange.png)

## Tasks

### `user-engagement`

Task is to predict if a user will make any engagement (vote, comment or post) on the site in the
next 2 years.

Filter: users who have made at least one comment/post/vote before the timestamp

ML task: Binary Classification

Evaluation Metric: AP

In [ ]:
task = get_task('rel-stack', 'user-engagement', download=True)

In [ ]:
task.get_table("train").df.columns

In [ ]:
print('TRAIN\n')

print(f'{len(task.get_table("train").df):,} samples over {task.get_table("train").df.OwnerUserId.nunique():,} users.')
print()
print('Label Distribution:')
print(task.get_table("train").df.contribution.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(task.get_table("train").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    task.get_table("train").df
    .drop_duplicates(subset='OwnerUserId', keep='last')
    .groupby('timestamp').OwnerUserId.nunique() / task.get_table("train").df.OwnerUserId.nunique()
)

In [ ]:
print('VAL\n')

print(f'{len(task.get_table("val").df):,} samples over {task.get_table("val").df.OwnerUserId.nunique():,} users.')
print()
print('Label Distribution:')
print(task.get_table("val").df.contribution.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(task.get_table("val").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print(f'User overlap between train and val: {len(set(task.get_table("train").df.OwnerUserId) & set(task.get_table("val").df.OwnerUserId)):,}')

**Notes**:

- `timestamp` is chopped up into two-year chunks rather than a finer grain (which would yield more samples per user)
- Train-val split is temporal
- Val is almost as large as train -- reflects the growth of the platform probably. Number of samples (& users) biased towards recent years.
- All users in train appear in val -- when evaluating look at performance diff b/w in-train and out-of-train user sets.

### `post-votes`

Task is to predict if the number of upvotes a question will get in the following 6 months.

ML task: Regression

Evaluation Metric: MAE

In [ ]:
task = get_task('rel-stack', 'post-votes', download=True)

In [ ]:
task.get_table("train").df.columns

In [ ]:
print('TRAIN\n')

print(f'{len(task.get_table("train").df):,} samples over {task.get_table("train").df.PostId.nunique():,} posts.')
print()
print('Temporal dist of samples:')
print(task.get_table("train").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of posts (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    task.get_table("train").df
    .drop_duplicates(subset='PostId', keep='last')
    .groupby('timestamp').PostId.nunique() / task.get_table("train").df.PostId.nunique()
)
print()
print('Label Distribution:')
mean = task.get_table("train").df.popularity.mean()
std = task.get_table("train").df.popularity.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Median: {task.get_table("train").df.popularity.quantile(0.5)}')
print(f'Prop zeros: {(task.get_table("train").df.popularity == 0).mean():.1%}')
fig = px.histogram(x=task.get_table("train").df.popularity + 1, log_x=True, log_y=True)
# ^-- add 1 so log scale shows
fig.update_layout(xaxis_title='Popularity', yaxis_title='Sample Count')

In [ ]:
print('VAL\n')

print(f'{len(task.get_table("val").df):,} samples over {task.get_table("val").df.PostId.nunique():,} posts.')
print()
print('Temporal dist of samples:')
print(task.get_table("val").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of posts (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    task.get_table("val").df
    .drop_duplicates(subset='PostId', keep='last')
    .groupby('timestamp').PostId.nunique() / task.get_table("val").df.PostId.nunique()
)
print()
print('Label Distribution:')
mean = task.get_table("val").df.popularity.mean()
std = task.get_table("val").df.popularity.std()
print(f'Mean (std): {mean:,.1f} ± {std:,.1f}')
print(f'Median: {task.get_table("val").df.popularity.quantile(0.5)}')
print(f'Prop zeros: {(task.get_table("val").df.popularity == 0).mean():.1%}')
fig = px.histogram(x=task.get_table("val").df.popularity + 1, log_x=True, log_y=True)
# ^-- add 1 so log scale shows
fig.update_layout(xaxis_title='Popularity', yaxis_title='Sample Count')

**Notes**:
- Label is incredibly skewed (vast majority of labels are 0)
- Will be hard to learn a good model --> could consider just doing classfication (0 vs nonzero) + some regression on the nonzero part

### `user-badge`

Task is to predict if the number of upvotes a question will get in the following 6 months.

ML task: Binary Classification

Evaluation Metric: AUC

In [ ]:
task = get_task('rel-stack', 'user-badge', download=True)

In [ ]:
task.get_table("train").df.columns

In [ ]:
print('TRAIN\n')

print(f'{len(task.get_table("train").df):,} samples over {task.get_table("train").df.UserId.nunique():,} users.')
print()
print('Label Distribution:')
print(task.get_table("train").df.WillGetBadge.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(task.get_table("train").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    task.get_table("train").df
    .drop_duplicates(subset='UserId', keep='last')
    .groupby('timestamp').UserId.nunique() / task.get_table("train").df.UserId.nunique()
)

In [ ]:
print('VAL\n')

print(f'{len(task.get_table("val").df):,} samples over {task.get_table("val").df.UserId.nunique():,} users.')
print()
print('Label Distribution:')
print(task.get_table("val").df.WillGetBadge.value_counts(normalize=True))
print()
print('Temporal dist of samples:')
print(task.get_table("val").df.timestamp.dt.strftime('%Y-%m').value_counts(normalize=True).sort_index())
print()
print('Temporal dist of users (first appearance):')
print(
    # timestamp is sorted in desc order ==> keep='last'
    task.get_table("val").df
    .drop_duplicates(subset='UserId', keep='last')
    .groupby('timestamp').UserId.nunique() / task.get_table("val").df.UserId.nunique()
)

## Raw Data

### Users

In [ ]:
users = dataset.get_db().table_dict['users'].df

In [ ]:
users.head()

In [ ]:
# Check NaNs
users.isna().mean()

In [ ]:
users.Location.apply(lambda x: str(x).split(',')[-1].strip()).value_counts(normalize=True)

In [ ]:
fig = px.histogram(x=users.CreationDate.dt.strftime('%Y-%m'))
fig.update_layout(height=500, width=700, title='Signups')

**Notes**:
- Some obvious features related to NaNs: eg nan aboutMe, nan location, nan website
- Use creation date for months_since_creation
- Would be nice to know when they edited profile last.
- Seasonal dip in signups in July/August (maybe b/c school's out!) -- check if seasonality extends to posts too

### Badges

In [ ]:
badges = dataset.get_db().table_dict['badges'].df

In [ ]:
badges.sample(n=15)

In [ ]:
badges.isna().mean()

In [ ]:
badges.TagBased.value_counts(normalize=True)

In [ ]:
badges.Name.value_counts(normalize=True)

In [ ]:
fig = px.histogram(x=badges.Date.dt.strftime('%Y-%m'))
fig.update_layout(height=500, width=700, title='New Badges')

In [ ]:
# Distribution of num tags
fig = px.histogram(x=badges.groupby('UserId').size(), log_x=True, log_y=True)
fig.update_layout(height=500, width=700, title='Dist. of Badge Counts',
                  xaxis_title='Number of Badges', yaxis_title='Frequency')

**Notes**
- Same seasonality observed (summer lull)
- The number of badges by user obeys a power law
- Can probably aggregate a users badges into a single feature that captures the sum total of their badges weighted by rarity
- Interesting to look at time since last badge and rarity of last badge by user

### Posts

In [ ]:
posts = dataset.get_db().table_dict['posts'].df
print(f'Total Posts: {len(posts):,}')

In [ ]:
posts.sample(n=15)

In [ ]:
posts.PostTypeId.value_counts(normalize=True)

In [ ]:
posts.ContentLicense.value_counts(normalize=True)

In [ ]:
posts.isna().mean()

In [ ]:
fig = px.histogram(x=posts.CreationDate.dt.strftime('%Y-%m'))
fig.update_layout(height=500, width=700, title='New Posts')

In [ ]:
fig = px.histogram(x=posts.groupby('OwnerUserId').size(), log_y=True, range_x=(0, 1000))
fig.update_traces(xbins=dict( # bins used for histogram
        start=0,
        end=1000,
        size=1
    ))
fig.update_layout(height=500, width=700, title='Dist. of Num Posts (per user)',
                  xaxis_title='Number of Posts', yaxis_title='Frequency (# Users)')

In [ ]:
num_answers = posts.groupby('ParentId').size()
# posts.set_index('Id', inplace=True)
posts.loc[num_answers.index, 'num_answers'] = num_answers.values
posts['num_answers'].fillna(0, inplace=True)

In [ ]:
fig = px.histogram(posts, x='num_answers', log_y=True, range_x=(0, 50))
fig.update_layout(height=500, width=700, title='Dist. of Num Answers (per post)',
                  xaxis_title='Number of Answers', yaxis_title='Frequency (# Posts)')

In [ ]:
posts['post_length'] = posts.Body.str.len()

In [ ]:
fig = px.histogram(posts, x='post_length', log_x=True, range_x=(50, 5_000))
fig.update_layout(height=500, width=700, title='Dist. of post length',
                  xaxis_title='Post Length', yaxis_title='Frequency (# Posts)')

In [ ]:
# posts per user over time
posts_per_user = (
    posts.groupby(posts.CreationDate.dt.strftime('%Y-%m')).size()
    / users.groupby(users.CreationDate.dt.strftime('%Y-%m')).size().cumsum()
)

In [ ]:
fig = px.line(x=posts_per_user.index, y=posts_per_user.values, width=700, height=500)
fig.update_layout(xaxis_title='Month', yaxis_title='Posts per User')

In [ ]:
def get_tag_list(tag_str: str) -> list[str]:
    return tag_str.strip('<>').split('><')

print(get_tag_list('<elementary-set-theory><intuition><infinity><faq>'))
print(get_tag_list('<computer-science>'))

In [ ]:
posts.Tags.head()

**Notes**
- Seasonality preserved.
- Power laws in number of posts (per user) and number of answers (per post)
- Post length has interesting dist (may be informative for churn?)
- Tags likely not informative (gut feeling is that signal will have been captured in other feats.)
- Post rate per user goes down because of churn (interesting to look at per active user w/ more time).
- Can focus solely on questions (type == 1) and answers (type == 2), other types are too rare.

### PostHistory

In [ ]:
post_hist = dataset.get_db().table_dict['postHistory'].df
print(f'Total Post History records: {len(post_hist):,}')

In [ ]:
post_hist.PostHistoryTypeId.value_counts(normalize=True)

In [ ]:
# posts closed or deleted
post_hist[post_hist.PostHistoryTypeId.isin({10, 12})].PostId.nunique()

In [ ]:
# post tweeted
post_hist[post_hist.PostHistoryTypeId == 25].PostId.nunique()

### Votes

In [ ]:
votes = dataset.get_db().table_dict['votes'].df
print(f'Total votes: {len(votes):,}')

In [ ]:
votes.isna().mean()

In [ ]:
# see full schema for type descriptions
votes['is_positive'] = votes.VoteTypeId.isin({1, 2, 5, 8, 16})
votes['is_negative'] = votes.VoteTypeId.isin({3, 4, 6, 10, 12})

In [ ]:
fig = go.Figure(
    [
            go.Histogram(x=votes[votes.is_positive].groupby('PostId').size(), name='positive', ),
            go.Histogram(x=votes[votes.is_negative].groupby('PostId').size(), name='negative')
    ],
    layout=dict(barmode='overlay')
)
# fig = px.histogram(x=votes[votes.is_positive].groupby('PostId').size(), log_y=True, range_x=(0, 100))
fig.update_traces(opacity=0.75)
fig.update_yaxes(type="log", range=[0, 6])
fig.update_xaxes(range=[0, 100])
fig.update_layout(height=500, width=700, title='Dist. of Votes (by Post)',
                  xaxis_title='Number of Votes', yaxis_title='Frequency (# Posts)')

In [ ]:
fig = go.Figure(
    [
            go.Histogram(x=votes[votes.is_positive].groupby('UserId').size(), name='positive'),
            go.Histogram(x=votes[votes.is_negative].groupby('UserId').size(), name='negative')
    ],
    layout=dict(barmode='overlay')
)
# fig = px.histogram(x=votes[votes.is_positive].groupby('PostId').size(), log_y=True, range_x=(0, 100))
fig.update_traces(opacity=0.75)
fig.update_yaxes(type="log", range=[0, 4])
fig.update_xaxes(range=[0, 100])
fig.update_layout(height=500, width=700, title='Dist. of Votes (by User)',
                  xaxis_title='Number of Votes', yaxis_title='Frequency (# User)')

In [ ]:
votes[votes.is_negative].UserId.isna().mean()

**Notes**:
- `UserId` is only present for `VoteTypeId` 5, 6, 7, 8 (unsure why) -- so user level feats only possible w/ positive votes
- "positive" engagements are more common (per post) than negative engagements
- Look at comment rate (ie comments per week/month)

### Comments

In [ ]:
comments = dataset.get_db().table_dict['comments'].df
print(f'Total comments: {len(comments):,}')

In [ ]:
comments.isna().mean()

In [ ]:
fig = px.histogram(x=comments.groupby('PostId').size(), log_y=True, range_x=(0, 50))
fig.update_layout(height=500, width=700, title='Dist. of Num Comments (per Post)',
                  xaxis_title='Number of Comments', yaxis_title='Frequency (# Posts)')

In [ ]:
fig = px.histogram(x=comments.groupby('UserId').size(), log_y=True, range_x=(0, 1000))
fig.update_layout(height=500, width=700, title='Dist. of Num Comments (per User)',
                  xaxis_title='Number of Comments', yaxis_title='Frequency (# Users)')